<img src="logo.png" alt="Vegeta" width="240">

# FALCO — the tractor mountain aircraft (35)

**Falco-Zero** is NISUS+ (notebook 33) rebuilt as the frame study (notebook 34) said: a **large tractor propeller**
in the nose, a **simple tail** on one tube, more efficiency. NISUS+'s wing (2.4 m, crow flaps and ailerons, the
three-piece spar), its 24 Li-ion cells, its Jetson Orin, its mountain missions and its bird hunts stay; the twin-boom
pusher pod becomes **MERLIN's round fuselage** with a spinner, a cowl, a cylinder and a tail cone, the **FALCO-109**
marking raised on both sides, a **T-Motor AT5220-A KV220** turning an **APC 20x15E**, the pack re-connected **8S3P**
(the same cells, the same box: the low-KV motor needs the volts), a conventional tail on a **roll-wrapped 30/28 carbon
tube** in a keel socket with one dorsal fin. The landing is a belly landing with the **propeller parked horizontal**: a
20-inch blade straight down would be 150 mm in the meadow.

This notebook is the **first engineering approximation**, as NISUS+'s: every number is calculated, sourced or labelled
as an assumption; missing analyses say **NOT RUN**. NISUS+'s modules do the arithmetic through backward-compatible
`design=` / `sim=` seams; FALCO's modules hold what the tractor changes.

| part | what | where the code is |
|---|---|---|
| 1 | requirements, the frame study's verdict, the starting shape | `designs/falco.py` |
| 2 | CAD (Dedalus): parts, three-view, **the parked propeller's clearance**, crow, transport, exploded view, packaging | `falco.py`, `falco_drawings.py` |
| 3 | materials, construction, load paths | `falco_systems.py`, `falco_structure.py` |
| 4 | the atmosphere; mass, CG (the motor in the nose against the pack), inertia | `falco_systems.py` |
| 5 | electronics and their energy; the propeller's position sensor | `falco_systems.py` |
| 6 | propulsion: the sourced motor's fit, **the drive options (why 8S)**, **the propeller trade (why 20x15)**, the drive over altitude, the brake region, the pack | `falco_systems.py`, `falco_flight.py` |
| 7 | aerodynamics: lattice with one fin, crow, derivatives, the tractor's installation effects, CFD at altitude (NOT RUN) | `falco_flight.py`, `falco_cfd.py` |
| 8 | structure: load cases (+ gyroscopic, fin torsion, blade strike), hand checks, FEA: spar, joiner, **tail tube A/B and modes**, keel socket, firewall, tray | `falco_structure.py` |
| 9 | flight against altitude and **NISUS+ vs FALCO** (speed, acceleration, climb, descent, regeneration) | `falco_flight.py` |
| 10 | mission energy over the mountains | `falco_systems.py` |
| 11 | MuJoCo (Chiron) missions with the propeller parked for the landing, movies; 11b the bird hunt | `falco_robot.py`, `falco_controller.py`, `falco_scenario.py`, `falco_birds.py` |
| 12 | the gain, the parked landing's requirement, the first recommended build, what to refine | — |

Switches: `VEGETA_DEBUG=1` mocks FEA and CFD; `FALCO_QUICK=1` flies only the calm mission and one bird hunt, small
movies; `VEGETA_SKIP_OPENFOAM=1` skips the CFD (NOT RUN); `FALCO_FEA=0` skips the FEA (NOT RUN).

In [ ]:
# DEBUG: test the notebook end to end before the full run (FEA and CFD mocked). On here or from the shell:
#   VEGETA_DEBUG=1 jupyter nbconvert --to notebook --execute 35_falco.ipynb --output 35_falco-run.ipynb
DEBUG = False
from vegeta import mock
DEBUG = mock.setup(DEBUG)

In [ ]:
import json, math, os, sys, time
from dataclasses import replace
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, Video, display
from vegeta import dedalus, talos, aeromant, boreas, cache
sys.path.insert(0, "designs")
import falco, falco_systems as fs, falco_flight as ff, falco_structure as fst, falco_cfd as fcfd, falco_drawings as fd
import falco_robot as fr, falco_controller as fc, falco_scenario as fsc, falco_birds as fb
import nisus_plus, nisus_plus_systems as nps, nisus_plus_flight as npf, frame_study as fstudy

pd.set_option("display.max_colwidth", 110); pd.set_option("display.width", 220)
ROOT = Path("_runs/falco"); ROOT.mkdir(parents=True, exist_ok=True)
OUT = Path("output") / "35_falco"; OUT.mkdir(parents=True, exist_ok=True)
for sub in ("cad", "drawings", "tables", "telemetry", "plots"):
    (OUT / sub).mkdir(exist_ok=True)
cache.notebook("35_falco")
QUICK = os.environ.get("FALCO_QUICK") == "1"
RUN_FEA = os.environ.get("FALCO_FEA", "1") != "0"
RUN_CFD = os.environ.get("VEGETA_SKIP_OPENFOAM") != "1"
G, RHO0 = fs.G, nps.RHO0
STATUS = {}
print(f"DEBUG {DEBUG} | QUICK {QUICK} | FEA {'on' if RUN_FEA else 'NOT RUN'} | CFD {'on' if RUN_CFD else 'NOT RUN'}")
def save_table(df, name):
    df.to_csv(OUT / "tables" / f"{name}.csv")
    return df

# Part 1 — requirements, the frame study's verdict, the starting shape

NISUS+'s requirements stand (the Alps up to 4500 m, climb ≥ 5 m/s at 4500 m, descent ≥ 10 m/s, a 20-min survey after
a 2800 m climb with 20 % reserve, the backpack), plus the frame study's three findings and the user's three decisions:

| from notebook 34 | what FALCO does |
|---|---|
| the 20-inch propeller does not fit between NISUS+'s booms (the spar fails, crow loses span) | the propeller goes to the **nose**: a tractor |
| a single tail is 213 g lighter, 3 x stiffer in bending, puts nothing into the wing; it must be **roll-wrapped** (torsion) | **one 30/28 roll-wrapped tube** in a keel socket, the fin on its end fitting |
| the 20-inch on a motor of the AT4125's class: −11 % mission energy, +20 % climb (an assumed motor) | a **real motor sourced** with its bench table (Part 6), its own gain measured against NISUS+ (Part 9) |

| decision | taken |
|---|---|
| the landing with a fixed nose propeller | **park it horizontal** with the ESC's brake and a position sensor before the flare; a keel; no gear (Part 2, 11) |
| the motor | the **AT5220 class, KV ~220**; the pack re-connected 8S3P when the 6S has not the volts (Part 6) |
| the fuselage | **MERLIN's tractor body** sized for the pack, the Orin and the sensors; the camera in the chin under the hub (image obstruction accepted: computer vision) |

The starting shape: NISUS+'s wing unchanged; the fuselage Ø130 x 830 mm, the spinner's tip 330 mm ahead of the wing's
leading edge (the motor in the nose against the pack under the wing: 28 % MAC); the tail at NISUS+'s arm (782 mm) with
its stabiliser (560 x 190) and one fin of 82 % of NISUS+'s two fins' area, on the centre line clear of the stabiliser's tips.

In [ ]:
P = falco.resolve()
L = falco.Falco.layout(P)
DES = falco.Falco()
pt = pd.DataFrame([{"parameter": q.name, "value": P[q.name], "unit": q.units, "description": q.description} for q in falco.Falco.parameters if q.name in P]).set_index("parameter")
display(save_table(pt, "parameters"))
display(pd.Series({k: L[k] for k in ("S_ref", "AR", "mac", "V_h", "V_v", "l_t", "n_fins", "prop_x", "x_pod_end", "x_tube0", "tube_free_length", "prop_ground_margin",
                                     "prop_ground_margin_resting", "prop_ground_margin_parked", "prop_ground_margin_parked_resting", "resting_pitch_deg",
                                     "overall_length", "longest_piece_mm")}, name="layout").to_frame())

# Part 2 — CAD (Dedalus)

In [ ]:
from tqdm.auto import tqdm
t0 = time.time()
GEO = DES.generate()
print(GEO.measure())
for part in tqdm(("aircraft", "wing", "pod", "nose", "tail_tube", "tail_socket", "tail_fitting", "tail", "motor_mount", "spar_joiner", "flap", "aileron"), desc="CAD parts (build, STEP)"):
    g = DES.generate(part=part)
    g.export_step(str(OUT / "cad" / f"falco_{part}.step"))
GEO.export_stl(str(OUT / "cad" / "falco_aircraft.stl"), tolerance=0.3)
CROW_GEO = DES.generate(flap_deg=55.0, aileron_deg=-25.0)
CROW_GEO.export_stl(str(OUT / "cad" / "falco_crow.stl"), tolerance=0.3)
print("crow pose:", CROW_GEO.measure()["valid"], CROW_GEO.measure()["n_solids"], "solid(s);", f"{time.time() - t0:.0f} s")
fig = fd.three_view(P); fig.savefig(OUT / "drawings" / "three_view.png", dpi=110); plt.show()

## The parked propeller

The blade-down clearance says why the propeller must be parked: a 20-inch blade straight down reaches 150 mm under the
keel's bottom. Parked horizontal, the lowest point is the spinner, 83 mm above the keel's bottom on a level touchdown and
more at the tail-low attitude of the rest (the keel's rear corner and the tail bumper carry the aircraft nose-up).

In [ ]:
PC, fig = fd.park_clearance(P); fig.savefig(OUT / "drawings" / "park_clearance.png", dpi=110); plt.show()
display(save_table(PC, "park_clearance"))
fig = fd.crow_view(P); fig.savefig(OUT / "drawings" / "crow.png", dpi=110); plt.show()
fig = fd.transport_view(P); fig.savefig(OUT / "drawings" / "transport.png", dpi=110); plt.show()
display(pd.Series(falco.transport_check(P)).to_frame("transport"))
try:
    fd.exploded_png(OUT / "drawings" / "exploded.png", P)
    display(Image(str(OUT / "drawings" / "exploded.png"), width=900))
except Exception as e:
    print("the exploded view needs pyvista off screen (xvfb-run):", e)

In [ ]:
fig = fd.internal_layout(fs.DEFAULT_PACK, P); fig.savefig(OUT / "drawings" / "internal_layout.png", dpi=110); plt.show()
display(pd.DataFrame(falco.Falco.bays(P), index=["x0 [mm]", "x1 [mm]", "width [mm]", "height [mm]"]).T)

# Part 3 — materials, construction and load paths

NISUS+'s wing as built (an XPS core with the 2 mm balsa D-box, the 20/17 carbon spar in three pieces with joiners, the
rear carry-through tube now 12/10: no booms' moment to carry). The fuselage is an LW-PLA printed shell in two pieces
(the nose cowl with the spinner's shell to the joint behind the firewall; the body to the tail cone), a solid PETG
**firewall** with the 52xx bolt pattern bonded in the cowl, the PETG keel frame with the wing saddle and the **tail
tube's socket** (ribbed, 200 mm from the saddle to the tail cone's end), the roll-wrapped 30/28 tube, its end fitting
carrying the stabiliser's 12/10 spar, the fin's rod and the bumper, the elevator and rudder servos on it. Load paths:
the lift → the spar → the saddle; the tail's loads → the tube → the socket (a couple on its two ends); the thrust and
the brake's reverse thrust → the firewall → the cowl; the landing → the keel and the tail bumper.

In [ ]:
display(pd.DataFrame(nps.MATERIALS).T)
display(pd.DataFrame(fs.structure_items(P), columns=["item", "group", "mass [g]", "x [mm]", "y [mm]", "z [mm]", "basis"]).set_index("item").round(1))

# Part 4 — the air, the mass, the centre of gravity, the inertia

The ISA with an offset as NISUS+'s. The mass table puts the 8S3P pack along its bay under the wing so the CG sits at
28 % MAC: with 600 g of motor, propeller and spinner 400 mm ahead of the wing, the nose had to be shortened to 330 mm
and the pack's bay moved aft to the tube's socket for the pack to reach it (the first placement sat at −2 % MAC).

In [ ]:
display(save_table(fs.atmosphere_table(), "atmosphere"))
MT = {k: fs.mass_table(k, p=P) for k in ("8s3p-p45b", "6s4p-p45b")}
CI = {k: fs.cg_inertia(t, P) for k, t in MT.items()}
display(save_table(MT[fs.DEFAULT_PACK].round(1), "mass_table"))
display(MT[fs.DEFAULT_PACK].groupby("group")["mass [g]"].sum().round(0).to_frame())
print("pack placement:", MT[fs.DEFAULT_PACK].attrs["ballast_note"] or "inside its bay at the CG target", MT[fs.DEFAULT_PACK].attrs["battery_x_range"])
display(pd.DataFrame({k: {x: c[x] for x in ("mass_kg", "x_cg_frac_mac", "Ixx", "Iyy", "Izz", "Ixz", "wing_loading_N_m2")} for k, c in CI.items()}).T.round(3))
display(fs.component_table()[["group", "mass [g]", "kind", "availability", "source"]])

# Part 5 — electronics and their energy

NISUS+'s loads on an 8S bus (every device rated to 36 V: the ESC's 3-8S rating, the flight controller's 6-36 V, a 36 V
buck and BEC), plus the **propeller's position sensor** (a Hall sensor on the firewall, a magnet on the spinner's
backplate: one pulse per turn) that the parking routine reads.

In [ ]:
display(save_table(fs.electrical_loads().round(2), "electrical_loads"))
display(fs.phase_power().round(1))
display(fs.servo_check(P).round(2))

# Part 6 — propulsion: the sourced motor, why 8S, why 20x15

The **T-Motor AT5220-A KV220** (52 mm stator, 450 g, 6-12S, 70 A / 2700 W for 180 s) is the one motor of the class
whose maker publishes a full bench table with the rpm: APC 18x8 at 12S from 40 to 100 % throttle (43.33 V, 48.86 A,
8141 rpm, 2.037 N m, 8185 gf at 100 %). The model is fitted to the 100 % point (R with the ESC and the wires, as NISUS+'s
fit); kt (I − I0) against the published torque checks the datasheet's I0; the 70 % row checks the propeller's torque
calibration. The calibration ratios are carried from the 18x8 to the 20-inch planform — assumed.

**Why 8S.** On the 6S pack the KV220 has the volts for ~3900 rpm on a 20x13: 0.5 kW shaft, static T/W below 0.7. The
KV380 variant on 6S would run the 20x13 past the ESC's 80 A and sits at that limit when held back. The same 24 cells
re-connected **8S3P** (the same box, the same 388 Wh) give the frame study's 20-inch point (~4800 rpm, ~1.1 kW shaft)
at ~50 A within every limit.

**Why 20x15.** The frame study's 20x13 cruises at 18 m/s EAS near its zero-thrust advance ratio: the blades sit at
their zero-lift angle and the profile drag is all that is left (67 % efficiency, more cruise power than NISUS+'s
15x8). The 20x15 cruises at 73 % and climbs 15 % faster for the same static current class; its pitch speed at the
static rpm (31 m/s) covers the dash.

In [ ]:
AF0 = {"mass_kg": CI[fs.DEFAULT_PACK]["mass_kg"], "cd0": falco.drag_buildup(P, 20.0)["cd0"], "AR": L["AR"], "oswald": 0.9, "S": L["S_ref"]}
REQ = fs.requirements(AF0["mass_kg"], AF0["cd0"], AF0["AR"], AF0["oswald"], AF0["S"])
display(save_table(REQ.round(2), "requirements"))
MOTOR, FIT = fs.motor_model()
display(pd.Series({k: v for k, v in FIT.items() if not isinstance(v, (dict, str))}, name="motor fit (AT5220-A KV220 on the 18x8 at 12S)").to_frame())
display(pd.DataFrame(fs.PUBLISHED_STATIC["rows"], columns=["throttle", "V", "A", "W", "rpm", "N m", "gf"]).set_index("throttle"))
DOPT = fs.drive_options(P, progress=True)
display(save_table(DOPT, "drive_options"))
print("selected:", DOPT.attrs["selected"], "—", DOPT.attrs["why"])

In [ ]:
A = ff.aero(P)
TRADE = ff.propeller_trade(p=P, progress=True)
display(save_table(TRADE.round(2), "propeller_trade"))
print("selected:", TRADE.attrs["selected"])
DR = fs.drive(rebuild=not Path(fs.PROPULSION_JSON).exists())
display(save_table(fs.drive_table(DR).round(2), "drive_vs_altitude"))

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4.2))
DRN = nps.drive()
for h, col in ((0, "k"), (3000, "C0"), (4500, "C3")):
    rho = fs.atmosphere(h)["rho"]
    V = np.linspace(0, 35, 36)
    ax[0].plot(V, [DR.max_thrust(v, rho) for v in V], color=col, label=f"FALCO {h} m")
    ax[0].plot(V, [DRN.max_thrust(v, rho) for v in V], color=col, ls=":", label=f"NISUS+ {h} m")
for V, col in ((15, "C2"), (25, "C1"), (30, "C4")):
    r = DR.rows(V)
    ax[1].plot(r["rpm"], r["thrust"], color=col, label=f"{V} m/s")
    ax[2].plot(r["rpm"], np.where(r["voltage"] > 0, r["voltage"] * r["current"], np.nan), color=col, label=f"{V} m/s")
    fw = DR.freewheel(V)
    ax[1].plot(fw["rpm"], fw["thrust"], "o", color=col)
ax[0].set(xlabel="airspeed [m/s]", ylabel="full-throttle thrust [N]", title="the drive against altitude (dotted: NISUS+'s 15x8)"); ax[0].legend(fontsize=7)
ax[1].axhline(0, color="k", lw=0.5); ax[1].set(xlabel="rpm", ylabel="thrust [N]", title="signed thrust of the 20x15 (dots: the free windmill)", ylim=(-20, 50)); ax[1].legend()
ax[2].axhline(0, color="k", lw=0.5); ax[2].set(xlabel="rpm", ylabel="motor terminal power [W]", title="negative: generating", ylim=(-400, 1000)); ax[2].legend()
for a in ax: a.grid(alpha=0.3)
fig.tight_layout(); fig.savefig(OUT / "plots" / "drive.png", dpi=110); plt.show()

## The Li-ion pack: 8S3P

NISUS+'s cells (Molicel P45B) in series-parallel 8S3P: 28.8 V nominal, 13.5 Ah, the same 388 Wh and the same
264 x 76 x 47 mm box; more resistance (46 mΩ against 28) at less current, the charge window 40 A (3 x 13.5).

In [ ]:
display(save_table(fs.pack_table().round(2), "packs"))
b = fs.pack()
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
soc = np.linspace(0, 1, 101)
for T, col in ((25, "k"), (5, "C0"), (-10, "C3")):
    ax[0].plot(soc, b.terminal_v(50.0, soc, T), color=col, label=f"50 A, {T} °C")
ax[0].plot(soc, b.ocv(soc), "k--", label="open circuit"); ax[0].set(xlabel="state of charge", ylabel="pack voltage [V]"); ax[0].legend(); ax[0].grid(alpha=0.3)
I = np.linspace(0, 100, 51)
for T, col in ((25, "k"), (-10, "C3")):
    ax[1].plot(I, I * b.terminal_v(I, 0.5, T), color=col, label=f"{T} °C, half charge")
ax[1].set(xlabel="current [A]", ylabel="power [W]"); ax[1].legend(); ax[1].grid(alpha=0.3)
fig.tight_layout(); plt.show()

# Part 7 — aerodynamics: the lattice, one fin, crow, the tractor's installation effects

NISUS+'s lattice on FALCO's planform (the same wing and stabiliser; the single fin enters the lateral derivatives at
the plate's own aspect ratio, its side force and its height above the tube) and the longer body in the pod's pitching
term. The **installation effects** of the nose propeller — MERLIN's open-propeller model (`air_propeller` + Boreas'
wake): the wake fraction and the thrust deduction at the flight points — are small for a tractor (the body sits behind
the disc) and enter the airframe's Cd0 at cruise; the slipstream's scrubbing of the fuselage and the wing root is **not**
in the build-up: the CFD rotor-disk case is its check (NOT RUN here).

In [ ]:
display(pd.Series({k: v for k, v in A.items() if isinstance(v, float)}, name="lattice").to_frame().round(4))
display(save_table(ff.crow_table(P, a=A).round(4), "crow_increments"))
CIZ = CI[fs.DEFAULT_PACK]
DER = ff.derivatives(P, x_cg_m=CIZ["x_cg_m"], z_cg_m=CIZ["z_cg_m"], a=A, h_m=3000.0)
display(save_table(DER, "derivatives"))
AF = {h: ff.airframe(fs.DEFAULT_PACK, P, h_m=h, a=A) for h in tqdm((0.0, 3000.0, 4500.0), desc="airframes (build-up + installation effects)")}
display(pd.DataFrame({h: {k: v for k, v in vars(a).items() if not isinstance(v, dict)} for h, a in AF.items()}).T)
display(save_table(ff.installation_table(AF[3000.0], DR, P), "installation_effects"))

## Whole-aircraft CFD at altitude (Aeromant / OpenFOAM) — NOT RUN

NISUS+'s cases in the ISA air of 3000 m (the α sweep, crow at V_FE against the clean aircraft) plus the tractor's own:
the 20x15 as a rotor disk **ahead of the nose** at the cruise rpm (the slipstream over the fuselage and the wing root),
and crow with the braked disk. No OpenFOAM in this session: the table lists them.

In [ ]:
atm3 = fs.atmosphere(3000.0)
V3 = ff.V_CRUISE_EAS / math.sqrt(atm3["sigma"])
CRUISE_RPM = DR.at(V3, DR.throttle_for_thrust(V3, AF[3000.0].drag(V3, atm3["rho"]), atm3["rho"]), atm3["rho"])["rpm"]
BRAKE_RPM = DR.brake(ff.V_FE_EAS / math.sqrt(atm3["sigma"]), 1.0, atm3["rho"])["rpm"]
if RUN_CFD:
    CFD_CASES = fcfd.plan(ROOT / "cfd", x_cg_m=CIZ["x_cg_m"], z_cg_m=CIZ["z_cg_m"], cruise_rpm=CRUISE_RPM, brake_rpm=BRAKE_RPM, fine=not QUICK)
    names = list(CFD_CASES)
    res = aeromant.run_cases([CFD_CASES[n] for n in names], jobs=1, processors=4, progress=True, cache=True)
    CFD = dict(zip(names, res))
    CFDT = fcfd.table(CFD, P)
else:
    CFD = {n: None for n in fcfd.specs(cruise_rpm=CRUISE_RPM, brake_rpm=BRAKE_RPM, fine=not QUICK)}
    CFDT = fcfd.not_run_table(cruise_rpm=CRUISE_RPM, brake_rpm=BRAKE_RPM)
STATUS["CFD"] = "run" if RUN_CFD and (CFDT["status"] == "solved").any() else "NOT RUN"
display(save_table(CFDT, "cfd_results"))

# Part 8 — structure

NISUS+'s load cases on FALCO's mass, drive and tail areas, plus the tractor's: the propeller's gyroscopic moment on the
firewall at the pull-out's pitch rate, the single tube's torsion from the one fin, and the **blade strike** — what the
meadow does to a 20-inch blade straight down at the touchdown speed: not a design case, the number that says why the
parking must work. The joints by hand: the keel socket's bearing and epoxy, the tube at its exit under the tail's lift
and the fin's side load together (bending + torsion with the roll-wrapped tube's G: the frame study's sizing term),
the stabiliser's 12/10 spar and the fin's rod on the end fitting (the first 8/6 rod came out at margin −0.03 under the
fin's ultimate side load: now 10/8), the firewall's screws, the straps, the flap horn. Talos: the spar and the joiner
(NISUS+'s cases), the **tail tube** A (symmetric pull-out) and B (the asymmetric gust) with the modal model on the frame
study's machinery (the saddle blocks, the sound-mesh retry), the **keel socket** (PETG) under the tube's bearing couple,
the **firewall** in flight, braking and landing, the battery tray.

In [ ]:
LC = fst.load_cases(P, dr=DR, a=A)
display(save_table(LC, "load_cases"))
display(save_table(fst.wing_hand(P, LC), "wing_hand"))
display(save_table(fst.joints_hand(P, LC), "joints_hand"))

In [ ]:
FEA_CASES = fst.models(ROOT / "fea", P, LC)
FEA = {}
if RUN_FEA:
    bar = tqdm(total=len(FEA_CASES), desc="FEA (spar, joiner, socket, firewall x3, tray)", bar_format="{desc}: {n_fmt}/{total_fmt} |{bar}| {elapsed} {postfix}")
for c in FEA_CASES:
    if RUN_FEA:
        bar.set_postfix_str(c.name)
    FEA[c.name] = c.model.ensure(ROOT / "fea" / c.name, run=RUN_FEA, threads=4)
    if RUN_FEA:
        bar.update(1)
if RUN_FEA:
    bar.close()
TT_CASES, TT = fst.solve_tail_tube(ROOT / "fea_tail", P, LC, run=RUN_FEA, threads=4, progress=RUN_FEA)
FEAT = pd.DataFrame([fst.summary_row(c, FEA[c.name]) for c in FEA_CASES] + [fst.summary_row(c, TT[c.name]) for c in TT_CASES if c.kind == "static"]).set_index("case")
STATUS["FEA"] = "run" if (FEAT["status"] == "solved").any() else "NOT RUN"
display(save_table(FEAT.T, "fea_results"))
display(save_table(fst.tail_tube_table(TT_CASES, TT, P), "tail_tube_fea"))
display(save_table(fst.tail_tube_modes(TT_CASES, TT), "tail_tube_modes"))
display(pd.DataFrame(fst.NOT_ESTABLISHED, columns=["not established", "why"]).set_index("not established"))

# Part 9 — flight against altitude, and NISUS+ against FALCO

Trim (with crow), the CG range (one fin, the long nose), the envelope at 0-6000 m, the ceiling, the climb strategy,
the descent in four configurations (the braked 20x15 is a bigger airbrake than the 15x8), the regeneration, the
downdraft escape, the launch, the limits — and the **side-by-side table**: static thrust, top speed, the acceleration
at the launch and at cruise, 15 → 25 m/s, the best climb, L/D, the cruise power at 1200 / 3000 / 4500 m, the
mission's energy and survey time, each aircraft at its design mass on its own drive map.

In [ ]:
AFW = AF[3000.0]
display(pd.DataFrame([ff.trim(DER, AFW.mass_kg, v / math.sqrt(fs.atmosphere(h)["sigma"]), fs.atmosphere(h)["rho"]) | {"h": h} for h in (0, 3000, 4500) for v in (16, 18, 22)]).round(2))
display(pd.DataFrame([ff.trim_crow(DER, AFW.mass_kg, 22.0, RHO0, 1.0, g) | {"γ": g} for g in (0, 15, 30)]).round(2))
CGR = ff.cg_range(P, AFW.mass_kg, a=A)
display(pd.Series(CGR).to_frame("CG range"))
EL = fs.phase_power()["electronics battery-side [W]"]
with tqdm(total=1, desc="envelope against altitude (0-6000 m)") as bar_env:
    ENV = ff.envelope_vs_altitude(AFW, DR, electronics_w=EL["survey"], battery_wh=fs.pack().energy_wh); bar_env.update(1)
display(save_table(ENV.round(2), "envelope_vs_altitude"))
display(pd.Series(ff.ceiling(AFW, DR)).to_frame("ceiling"))
display(pd.Series(ff.ceiling(AFW, DR, dT=20.0)).to_frame("ceiling, ISA +20"))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
AFN = npf.airframe(nps.DEFAULT_PACK, h_m=3000.0)
for h, col in ((0, "k"), (1500, "C0"), (3000, "C2"), (4500, "C3"), (6000, "C4")):
    e = ff.envelope(AFW, DR, h)
    en = npf.envelope(AFN, DRN, h)
    ax[0].plot(e["V"], e["roc"], color=col, label=f"FALCO {h} m"); ax[0].plot(en["V"], en["roc"], color=col, ls=":", lw=0.9)
    ax[1].plot(e["V"], e["P_level"], color=col, label=f"FALCO {h} m"); ax[1].plot(en["V"], en["P_level"], color=col, ls=":", lw=0.9)
ax[0].axhline(5.0, color="r", ls=":", label="5 m/s target"); ax[0].set(xlabel="TAS [m/s]", ylabel="climb rate [m/s]", title="the climb against altitude (dotted: NISUS+)")
ax[1].set(xlabel="TAS [m/s]", ylabel="level power, battery side [W]", ylim=(0, 600), title="level flight (dotted: NISUS+)")
for a in ax: a.grid(alpha=0.3); a.legend(fontsize=8)
fig.tight_layout(); fig.savefig(OUT / "plots" / "envelope.png", dpi=110); plt.show()
CS = ff.climb_strategy(DR, AFW)
display(save_table(CS.round(2), "climb_strategy"))

In [ ]:
DT = ff.descent_table(AFW, DR, DER, 3000.0)
display(save_table(DT.round(2), "descent_3000m"))
best = ff.best_descent(DT)
display(save_table(best[["EAS [m/s]", "TAS [m/s]", "sink [m/s]", "path angle [deg]", "battery side [W]"]].round(2), "best_descent"))
fig, ax = plt.subplots(figsize=(8, 4.5))
for name, g in DT.groupby("configuration"):
    ax.plot(g["EAS [m/s]"], g["sink [m/s]"], "o-", label=name)
ax.axhline(10, color="r", ls=":", label="10 m/s target"); ax.axvline(ff.V_FE_EAS, color="k", ls="--", lw=0.8, label="V_FE (crow)")
ax.set(xlabel="EAS [m/s]", ylabel="sink rate [m/s]", title="the descent at 3000 m"); ax.grid(alpha=0.3); ax.legend(fontsize=8)
fig.tight_layout(); fig.savefig(OUT / "plots" / "descent.png", dpi=110); plt.show()
REGEN = ff.regen_table(AFW, DR, DER, pk=fs.pack())
display(save_table(REGEN.round(3), "regeneration"))
display(save_table(ff.downdraft_escape(AFW, DR).round(2), "downdraft_escape"))
display(save_table(ff.launch_table(AFW, DR, a=A, p=P).round(2), "launch"))
display(save_table(ff.operating_limits(AFW, DR, DER, CGR), "operating_limits"))

## NISUS+ vs FALCO

In [ ]:
CMPX = ff.compare_with_nisus_plus(af_f=ff.airframe(fs.DEFAULT_PACK, P, h_m=1200.0, a=A), dr_f=DR, dr_n=DRN)
display(save_table(CMPX, "nisus_plus_vs_falco"))
num = CMPX.apply(pd.to_numeric, errors="coerce").dropna()
rows = [r for r in num.index if any(k in r for k in ("static thrust", "top speed", "acceleration at 17", "best climb", "cruise power", "mission energy used", "feasible survey", "drag area"))]
fig, ax = plt.subplots(figsize=(11, 5))
ax.barh(rows, num.loc[rows, "FALCO vs NISUS+ [%]"], color=["#2ca02c" if (v > 0) == ("power" not in r and "energy" not in r and "drag" not in r) else "#d62728" for r, v in zip(rows, num.loc[rows, "FALCO vs NISUS+ [%]"])])
ax.axvline(0, color="k", lw=0.8); ax.set(xlabel="FALCO against NISUS+ [%] (green: better)", title="speed, acceleration, climb, power, energy"); ax.grid(alpha=0.3, axis="x")
fig.tight_layout(); fig.savefig(OUT / "plots" / "nisus_plus_vs_falco.png", dpi=110); plt.show()

# Part 10 — mission energy over the mountains

NISUS+'s sizing mission (300 s on the meadow at 1200 m, the climb to 4000 m at 6 m/s, 3 km out, 20 min of survey,
back, the crow + brake descent at 10 m/s with its regeneration, the approach; 20 % reserve) on FALCO's drive and
airframe; the battery iteration shows the 8S3P against the 6S4P on the same aircraft (the same energy; the 6S4P would
need the KV380 at the ESC's limit — Part 6).

In [ ]:
REGEN_W = -float(DT[(DT["configuration"] == "crow + propeller brake") & (DT["EAS [m/s]"] == 20.0)]["battery side [W]"].iloc[0])
PLAN = fs.MissionPlan(work_alt_m=4000.0, descent_regen_w=REGEN_W)
afd = ff.airframe_dict(AFW)
ME = fs.mission_energy(fs.pack(), DR, afd, PLAN)
display(save_table(ME["phases"].round(2), "mission_energy"))
display(pd.Series({k: ME[k] for k in ("E_nominal_wh", "E_available_wh", "E_used_wh", "E_left_pct_available", "E_reserve_wh", "reserve_kept", "trigger_wh",
                                      "t_climb_s", "E_climb_wh", "potential_energy_wh", "climb_efficiency", "t_survey_feasible_s")}).to_frame("mission"))
MTAB = fs.mission_table(DR, lambda k: ff.airframe_dict(ff.airframe(k, P, h_m=3000.0, a=A)), PLAN)
display(save_table(MTAB, "battery_iteration"))
MEN = nps.mission_energy(nps.pack(), DRN, npf.airframe_dict(AFN), PLAN)
print(f"NISUS+ on the same plan: {MEN['E_used_wh']:.0f} Wh used, survey up to {MEN['t_survey_feasible_s'] / 60:.0f} min; FALCO: {ME['E_used_wh']:.0f} Wh, {ME['t_survey_feasible_s'] / 60:.0f} min")
COLD = fs.mission_energy(fs.pack(), DR, afd, replace(PLAN, pack_T_C=-5.0, dT=-10.0, descent_regen_w=0.0))
print(f"a cold morning (pack −5 °C, ISA −10): available {COLD['E_available_wh']:.0f} Wh, used {COLD['E_used_wh']:.0f} Wh, left {COLD['E_left_pct_available']:.0f} %, "
      f"reserve kept: {COLD['reserve_kept']}")

# Part 11 — the missions in MuJoCo over the mountains (Chiron), the propeller parked for the landing

NISUS+'s simulation (one rigid body with the mass table's rows, `FalcoAero` — NISUS+'s hook — with the density from the
ISA at the aircraft's altitude, the drive at the pack's loaded voltage, the brake that regenerates; the `Massif` and the
`MountainWind`; the TECS-style controller) with FALCO's robot, drive and derivatives. New: the **park** command. On the
9° crow final below 20 m the `FalcoController` sets the throttle to idle and the brake on and commands the park: the
ESC's active brake stops the blades and the parking routine (the Hall sensor, the nudge: 1 s assumed, no rotor inertia
in the model) marks them horizontal — `parked` in the log; the flare is flown on the pitch alone; a go-around clears
the park and the motor spins up. The judge counts a touchdown with the blades not parked as a **blade strike**.

In [ ]:
lab = fsc.make_lab(fsc.scenario("calm"), log_geoms=False)
display(pd.Series(fr.inertia_check(lab, lab.aero.robot.mass_table)).to_frame("MuJoCo vs the mass table"))
print("the hook's density at 3000 m:", lab.aero.air(3000.0)[0], "(ISA:", fs.atmosphere(3000.0)["rho"], ")")
M = fr.Massif()
X, Y = np.meshgrid(np.linspace(-3000, 7000, 300), np.linspace(-4000, 5500, 300))

In [ ]:
SCNS = [fsc.scenario("calm")] if QUICK else fsc.standard_scenarios()
EPS = []
for scn in tqdm(SCNS, desc="mountain missions (MuJoCo)"):            # one bar over the scenarios; one per mission over its simulated time
    t0 = time.time()
    labs = fsc.make_lab(scn, log_geoms=True)
    ep = fsc.run(labs, scn, progress=True)
    EPS.append((scn, ep))
    fsc.timeseries(ep).to_csv(OUT / "telemetry" / f"{scn.slug}.csv", index=False)
    print(f"{scn.label:48s} {ep.outcome['reason']:20s} {ep.outcome['detail']}  ({time.time() - t0:.0f} s)")
    for e in ep.log["events"]:
        if e[1] in ("weather", "jetson", "energy", "companion lost", "landing", "descent", "propeller") and "charged" not in e[2]:
            print(f"     {e[0]:7.1f} s  {e[1]}: {e[2]}")
CMP = fsc.compare([e for _, e in EPS])
display(save_table(CMP, "missions"))
display(save_table(fsc.park_table([e for _, e in EPS]), "parking"))

In [ ]:
scn, ep = EPS[0]
display(save_table(fsc.phase_table(ep).round(2), f"phases_{scn.slug}"))
display(save_table(fsc.energy_table(ep).round(2), f"energy_{scn.slug}"))
fig, ax = plt.subplots(2, 2, figsize=(15, 8))
for scn_, ep_ in EPS:
    ts = fsc.timeseries(ep_)
    ax[0, 0].plot(ts["t"], ts["z_asl"], label=scn_.name)
    ax[0, 1].plot(ts["t"], ts["vz"], lw=0.7, label=scn_.name)
    ax[1, 0].plot(ts["t"], ts["E_used_Wh"], label=scn_.name)
    ax[1, 1].plot(ts["x"], ts["y"], lw=0.8, label=scn_.name)
ts = fsc.timeseries(EPS[0][1])
ax[0, 0].plot(ts["t"], ts["z_asl"] - ts["agl"], "k:", lw=0.8, label="the ground below (calm)")
ax[0, 0].set(ylabel="altitude [m]"); ax[0, 1].set(ylabel="climb rate [m/s]"); ax[1, 0].set(xlabel="t [s]", ylabel="energy used [Wh]")
ax[1, 1].contour(X, Y, M.height(X, Y), levels=np.arange(1500, 4400, 250), colors="0.75", linewidths=0.5); ax[1, 1].set_aspect("equal"); ax[1, 1].set(xlim=(-1500, 4000), ylim=(-1500, 3500), title="tracks")
for a in ax.flat: a.grid(alpha=0.3)
ax[0, 0].legend(fontsize=7)
fig.tight_layout(); fig.savefig(OUT / "plots" / "missions.png", dpi=110); plt.show()
# the last 40 s of the calm landing: the park command, the rpm, the height
fig, ax = plt.subplots(1, 3, figsize=(15, 3.8))
td = ep.log["touchdown"] or ts["t"].iloc[-1]
w = ts[(ts["t"] > td - 40) & (ts["t"] < td + 5)]
ax[0].plot(w["t"], w["agl"]); ax[0].axvline(ep.log["t_park_cmd"], color="C1", ls="--", label="park commanded"); ax[0].axvline(ep.log["t_parked"] or td, color="C2", ls="--", label="parked"); ax[0].set(ylabel="agl [m]", xlabel="t [s]"); ax[0].legend(fontsize=8)
ax[1].plot(w["t"], w["rpm"]); ax[1].set(ylabel="propeller rpm", xlabel="t [s]")
ax[2].plot(w["t"], w["EAS"], label="EAS"); ax[2].plot(w["t"], 100 * w["parked"], label="parked x 100"); ax[2].plot(w["t"], 100 * w["brake"], label="brake x 100"); ax[2].set(xlabel="t [s]"); ax[2].legend(fontsize=8)
for a in ax: a.grid(alpha=0.3)
fig.suptitle("the calm landing: the propeller parked on the final"); fig.tight_layout(); fig.savefig(OUT / "plots" / "parking.png", dpi=110); plt.show()

In [ ]:
MOVIES = []
for scn, ep in (EPS[:1] if QUICK else EPS):
    path = OUT / f"{scn.slug}.mp4"
    try:
        fsc.render_movie(ep, scn, path, speed=30.0, fps=20, size=(640, 360) if QUICK else (960, 540), progress=True)
        MOVIES.append(path); print("written", path)
    except Exception as e:
        print(f"{scn.label}: the movie needs pyvista off screen (xvfb-run): {type(e).__name__}: {e}")
if MOVIES:
    display(Video(str(MOVIES[0]), embed=False, width=800))

## Part 11b — chasing birds over the slopes

NISUS+'s four hunts (notebook 33, Part 11b: golden eagles beating along the face in the ridge lift, griffon vultures
circling, the alpine chough flock, the eagles with the Jetson failing) on Falco-Zero. The camera sits in the chin under
the propeller's hub: the blades cross the top of the frame and computer vision masks them (an rpm-synchronous pattern) —
accepted, not modelled. The hunt ends with the propeller parked for the landing as in Part 11.

In [ ]:
BSS = [fb.bird_scenarios()[0]] if QUICK else fb.bird_scenarios()
BEPS = []
for bs in tqdm(BSS, desc="bird hunts (MuJoCo + vegeta.mission)"):
    t0 = time.time()
    ep = fb.run(bs, log_geoms=True, progress=True)
    BEPS.append(ep)
    fb.photo_table(ep).to_csv(OUT / "tables" / f"{bs.slug}_photos.csv", index=False)
    print(f"{bs.name:34s} {ep.outcome['reason']:18s} ({time.time() - t0:.0f} s)")
BCMP = fb.compare(BEPS)
display(save_table(BCMP.T, "bird_hunts"))
display(fb.photo_table(BEPS[0]).head(15))

In [ ]:
BMOVIES = []
for ep in BEPS[:1]:
    path = OUT / f"{ep.bird_scenario.slug}.mp4"
    try:
        fb.render_movie(ep, path, progress=True)
        BMOVIES.append(path); print("written", path)
    except Exception as e:
        print(f"the bird video needs pyvista off screen (xvfb-run): {type(e).__name__}: {e}")
if BMOVIES:
    display(Video(str(BMOVIES[0]), embed=False, width=900))

# Part 12 — the gain, the parked landing, the first recommended build

**The gain.** Against NISUS+ on the same mission (Part 9's table): the static thrust, the climb and the acceleration up
by the 20-inch disc, the cruise power down a little (the 20x15's efficiency against the longer body's drag), the
mission's energy down and the survey time up. The frame study's −11 % mission energy was with a motor of the AT4125's
loss class wound for the 20-inch propeller; the sourced AT5220's fitted resistance (0.13 Ω with the ESC) and its idle
current take part of it back — a lighter motor of that class with a published table (todo 5p.6) is the next step.

**The parked landing.** The propeller must be parked: a blade down is 150 mm in the meadow, and every mission above
lands with the blades horizontal — in the model, where the stop and the nudge take an assumed second. What is not
established: the ESC's brake-on-stop angle and the nudge's repeatability on the bench (todo 5p.4); until then a landing
with the blades not parked is the blade strike the judge names.

In [ ]:
gain = CMPX["FALCO vs NISUS+ [%]"]
BUILD = pd.Series({
    "airframe": "2.4 m tractor, MERLIN's round fuselage (LW-PLA, FALCO-109), NISUS+'s three-piece wing with crow flaps and ailerons, one 30/28 roll-wrapped tail tube, a dorsal fin",
    "mass (8S3P)": f"{CI['8s3p-p45b']['mass_kg']:.2f} kg, CG 28 % MAC, wing loading {CI['8s3p-p45b']['wing_loading_N_m2']:.0f} N/m²",
    "drive": "T-Motor AT5220-A KV220 + APC 20x15E (fixed; parked for the landing), 80 A 8S AM32-class ESC with active braking, a Hall position sensor",
    "pack": f"8S3P Molicel P45B ({fs.pack().energy_wh:.0f} Wh, {fs.pack().mass_g:.0f} g) in an insulating sleeve",
    "computer": "Jetson Orin Nano Super on a 36 V-rated 12 V buck, IMX477 + 6 mm in the chin, LTE link; the DEM of the area onboard",
    "avionics": "H743-WING class, pitot-static airspeed (EAS) under the chin, lidar for the flare, ELRS 900 MHz, RFD900x",
    "launch / landing": "a light bungee (17 m/s) / a 9° crow approach, the propeller parked below 20 m, a belly landing on the keel",
    "climb (MTOW)": f"{ENV.loc['0 m', 'climb max [m/s]']:.1f} m/s at sea level, {ENV.loc['4500 m', 'climb max [m/s]']:.1f} m/s at 4500 m",
    "fast descent at 3000 m": f"{best.loc['crow + propeller brake', 'sink [m/s]']:.1f} m/s (crow + brake at {best.loc['crow + propeller brake', 'EAS [m/s]']:.0f} m/s EAS)",
    "mountain mission (Part 10)": f"{ME['E_used_wh']:.0f} Wh of {ME['E_available_wh']:.0f} available, {ME['E_left_pct_available']:.0f} % left, survey up to {ME['t_survey_feasible_s'] / 60:.0f} min",
    "against NISUS+": f"static thrust {gain['static thrust, sea level [N]']:+.0f} %, climb at 3000 m {gain['best climb at 3000 m [m/s]']:+.0f} %, cruise power at 3000 m {gain['cruise power 18 m/s EAS, 3000 m, battery side [W]']:+.0f} %, "
                      f"mission energy {gain['mission energy used [Wh]']:+.0f} %, survey time {gain['mission: feasible survey [min]']:+.0f} %",
}, name="the first recommended build")
display(BUILD.to_frame())

## Assumptions to refine next

- **The motor's point on the 20x15 at 8S**: the fit carries the 18x8's calibration to the 20-inch planform; a thrust
  stand run, and the real APC blade in Boreas.
- **The parking**: the ESC's brake-on-stop angle, the nudge, the Hall sensor, the companion script's timing against the
  20 m park height; the stop's duration (1 s assumed, no rotor inertia in the model).
- **The slipstream** over the fuselage and the wing root (drag, the tail's dynamic pressure at power, the pitot's
  reading): the CFD rotor-disk case, then flight tests.
- **Flutter at TAS** with the single tube: the roll-wrapped tube's G measured, Talos modes with the real G.
- **The fin's rod** at margin ~0.9 (10/8): a carbon plate fin would be lighter and stiffer.
- NISUS+'s own list: the brake region of the propeller, crow's drag, the Li-ion model in the cold, the mountain wind,
  the launch, the bird species, the terrain.

In [ ]:
STATUS.update({"CAD": "run", "drawings": "run", "mass budgets": "run", "electronics": "run", "propulsion (sourced motor, drive options, propeller trade)": "run",
               "vortex lattice + crow + installation effects": "run", "mission energy": "run",
               "MuJoCo missions": f"run ({len(EPS)} scenarios, {sum(e.outcome['success'] for _, e in EPS)} landed, {sum(bool(e.log.get('parked_at_touchdown')) for _, e in EPS)} with the propeller parked)",
               "bird hunts": f"run ({len(BEPS)})", "movies": f"{len(MOVIES) + len(BMOVIES)} written"})
if DEBUG:
    STATUS["FEA"] = STATUS["CFD"] = "MOCKED (VEGETA_DEBUG): the numbers mean nothing"
display(pd.Series(STATUS, name="status").to_frame())

In [ ]:
def plain(x):
    if isinstance(x, (np.floating, float)):
        return None if not np.isfinite(x) else float(x)
    if isinstance(x, (np.integer,)):
        return int(x)
    if isinstance(x, (np.bool_,)):
        return bool(x)
    return x
DESIGN = {"source": "35_falco", "params": {k: plain(v) for k, v in P.items()}, "layout": {k: plain(v) for k, v in L.items() if not isinstance(v, dict)},
          "mass": {k: {kk: plain(x) for kk, x in c.items() if kk != "inertia"} for k, c in CI.items()},
          "derivatives": {k: plain(x) for k, x in DER["value"].items()}, "motor_fit": {k: plain(x) for k, x in FIT.items() if not isinstance(x, (str, dict))},
          "drive_options": {k: {kk: plain(vv) for kk, vv in row.items()} for k, row in DOPT.T.to_dict().items()},
          "propeller_trade": {k: {kk: plain(vv) for kk, vv in row.items()} for k, row in TRADE.T.to_dict().items()},
          "envelope": {h: {k: plain(x) for k, x in row.items()} for h, row in ENV.T.to_dict().items()},
          "mission_energy": {k: plain(ME[k]) for k in ("E_used_wh", "E_left_pct_available", "reserve_kept", "t_survey_feasible_s", "trigger_wh")},
          "nisus_plus_vs_falco": {k: {kk: plain(vv) for kk, vv in row.items()} for k, row in CMPX.T.to_dict().items()},
          "missions": {k: {kk: plain(vv) for kk, vv in row.items()} for k, row in CMP.T.to_dict().items()},
          "parking": {k: {kk: plain(vv) for kk, vv in row.items()} for k, row in fsc.park_table([e for _, e in EPS]).T.to_dict().items()},
          "fea": {k: {kk: plain(vv) for kk, vv in row.items()} for k, row in FEAT.T.to_dict().items()},
          "cfd": {k: {kk: plain(vv) for kk, vv in row.items()} for k, row in CFDT.T.to_dict().items()}, "status": STATUS}
target = OUT / "falco_design.json" if DEBUG else Path("designs/data/falco_design.json")
target.write_text(json.dumps(DESIGN, indent=1, default=str))
print("written", target)